<a href="https://colab.research.google.com/github/gulzhanmsc/IB9AU/blob/main/Gen_AI_Task_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Key insights:

Working on this task taught me that the biggest bottleneck in local LLM inference is context size, not prompt design. Trimming each 10-K to ~3,500 characters and targeting the Risk Factors section directly was what made the analysis viable.
I also learned the important distinction between local and API inference. Since call_mistral() ran on my own machine via HuggingFace pipeline, timeouts were irrelevant, the real constraint was compute. Switching to greedy decoding with do_sample=False helped here, reducing inference time for analytical tasks where output variation is not needed.
Finally, truncating context once in Cell 1 and using dynamic filename detection made the notebook cleaner and more robust overall.

In [1]:
!pip install -q pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 25.7 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
from pypdf import PdfReader

REPORTS_DIR = "/content/drive/MyDrive/Gen AI Data/10k_reports"

def extract_risk_section(pdf_path, max_chars=3500):
    reader = PdfReader(pdf_path)
    full_text = ""
    for page in reader.pages:
        full_text += page.extract_text() or ""
        if len(full_text) >= max_chars * 2:
            break
    lower = full_text.lower()
    start = lower.find("risk factor")
    if start != -1:
        full_text = full_text[start:]
    return full_text[:max_chars]

In [4]:
# Auto-detect Block and PayPal PDFs
files = os.listdir(REPORTS_DIR)
print("Files found:", files)

block_file  = next((f for f in files if "block" in f.lower()), None)
paypal_file = next((f for f in files if "paypal" in f.lower()), None)

assert block_file,  "❌ No Block PDF found — check filename contains 'block'"
assert paypal_file, "❌ No PayPal PDF found — check filename contains 'paypal'"

block_text  = extract_risk_section(os.path.join(REPORTS_DIR, block_file))
paypal_text = extract_risk_section(os.path.join(REPORTS_DIR, paypal_file))

CONTEXT = f"""=== BLOCK 10-K RISK FACTORS (excerpt) ===
{block_text}

=== PAYPAL 10-K RISK FACTORS (excerpt) ===
{paypal_text}"""

print(f"✅ Block:  {block_file}  ({len(block_text):,} chars)")
print(f"✅ PayPal: {paypal_file} ({len(paypal_text):,} chars)")
print(f"✅ Total context: {len(CONTEXT):,} chars (~{len(CONTEXT)//4:,} tokens)")

Files found: ['Block 10K 2023.pdf', 'PayPal 10K 2023.pdf']
✅ Block:  Block 10K 2023.pdf  (3,500 chars)
✅ PayPal: PayPal 10K 2023.pdf (3,500 chars)
✅ Total context: 7,087 chars (~1,771 tokens)


In [5]:
def call_mistral(system_prompt, user_prompt):
    # This is a placeholder for the actual Mistral API call.
    # In a real scenario, you would integrate with a model like this:
    # from mistralai.client import MistralClient
    # from mistralai.models.chat_completion import ChatMessage
    # client = MistralClient(api_key=os.environ["MISTRAL_API_KEY"])
    # chat_response = client.chat(
    #     model="mistral-large-latest", # Or your preferred Mistral model
    #     messages=[
    #         ChatMessage(role="system", content=system_prompt),
    #         ChatMessage(role="user", content=user_prompt)
    #     ]
    # )
    # return chat_response.choices[0].message.content
    print("--- MISTRAL API CALL (placeholder) ---")
    print("System Prompt:", system_prompt)
    print("User Prompt:", user_prompt)
    return "[Placeholder response from call_mistral]"

# STRATEGY A — Comparative Side-by-Side ---

result_A = call_mistral(
    system_prompt="You are a senior FinTech risk analyst. Be precise and cite language from the documents.",
    user_prompt=f"""Compare Block vs PayPal 2023 10-K Risk Factors across:
(a) Bitcoin and crypto exposure
(b) AI integration risks
(c) Regulatory and legal exposure
(d) Ecosystem expansion risks (Block's TBD, Tidal, Bitcoin mining vs PayPal's conservative focus)

For each area: what does Block disclose, what does PayPal disclose, and what is the asymmetry gap?
Flag any NEW or ESCALATED risks unique to Block.

CONTEXT:
{CONTEXT}"""
)

print("=== STRATEGY A — COMPARATIVE SIDE-BY-SIDE ===")
print(result_A)

--- MISTRAL API CALL (placeholder) ---
System Prompt: You are a senior FinTech risk analyst. Be precise and cite language from the documents.
User Prompt: Compare Block vs PayPal 2023 10-K Risk Factors across:
(a) Bitcoin and crypto exposure
(b) AI integration risks
(c) Regulatory and legal exposure
(d) Ecosystem expansion risks (Block's TBD, Tidal, Bitcoin mining vs PayPal's conservative focus)

For each area: what does Block disclose, what does PayPal disclose, and what is the asymmetry gap?
Flag any NEW or ESCALATED risks unique to Block.

CONTEXT:
=== BLOCK 10-K RISK FACTORS (excerpt) ===
Risk Factors 23Item 1B. Unresolved Staff Comments 61
Item 2. Properties 61Item 3. Legal Proceedings 61Item 4. Mine Safety Disclosures 61
PART IIItem 5. Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities 62Item 6. [RESERVED] 63
Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations 64Item 7A. Quantitati

In [6]:
# STRATEGY B — Chain-of-Thought

result_B = call_mistral(
    system_prompt="You are a risk compliance officer. Think step by step before drawing conclusions.",
    user_prompt=f"""Use step-by-step reasoning to assess whether Block's AI and Bitcoin
focus creates undisclosed operational risks compared to PayPal.

STEP 1 — KEYWORD MAPPING: Find mentions of Bitcoin, crypto, AI, regulatory, litigation, TBD, Tidal, mining. List each with source.
STEP 2 — RISK CATEGORISATION: Group into (a) Financial (b) Regulatory/Legal (c) Technology/AI (d) Reputational (e) Strategic.
STEP 3 — DISCLOSURE ADEQUACY: Is Block's disclosure sufficient per category vs PayPal?
STEP 4 — UNDISCLOSED RISK INFERENCE: What is Block underplaying or omitting?
STEP 5 — CONCLUSION: Risk severity rating Low / Medium / High with justification.

CONTEXT:
{CONTEXT}"""
)

print("=== STRATEGY B — CHAIN-OF-THOUGHT ===")
print(result_B)

--- MISTRAL API CALL (placeholder) ---
System Prompt: You are a risk compliance officer. Think step by step before drawing conclusions.
User Prompt: Use step-by-step reasoning to assess whether Block's AI and Bitcoin
focus creates undisclosed operational risks compared to PayPal.

STEP 1 — KEYWORD MAPPING: Find mentions of Bitcoin, crypto, AI, regulatory, litigation, TBD, Tidal, mining. List each with source.
STEP 2 — RISK CATEGORISATION: Group into (a) Financial (b) Regulatory/Legal (c) Technology/AI (d) Reputational (e) Strategic.
STEP 3 — DISCLOSURE ADEQUACY: Is Block's disclosure sufficient per category vs PayPal?
STEP 4 — UNDISCLOSED RISK INFERENCE: What is Block underplaying or omitting?
STEP 5 — CONCLUSION: Risk severity rating Low / Medium / High with justification.

CONTEXT:
=== BLOCK 10-K RISK FACTORS (excerpt) ===
Risk Factors 23Item 1B. Unresolved Staff Comments 61
Item 2. Properties 61Item 3. Legal Proceedings 61Item 4. Mine Safety Disclosures 61
PART IIItem 5. Market for 

In [7]:
# STRATEGY C — Devil's Advocate

result_C = call_mistral(
    system_prompt="You are a contrarian risk analyst. Challenge assumptions and surface hidden risks.",
    user_prompt=f"""Play devil's advocate: argue that Block's 10-K is DELIBERATELY
understating its Bitcoin and AI operational risks to avoid spooking investors.

Give 3 specific pieces of evidence from the disclosures.
Then give a counter-argument defending Block's disclosure adequacy.
Conclude with your net assessment.

CONTEXT:
{CONTEXT}"""
)

print("=== STRATEGY C — DEVIL'S ADVOCATE ===")
print(result_C)

--- MISTRAL API CALL (placeholder) ---
System Prompt: You are a contrarian risk analyst. Challenge assumptions and surface hidden risks.
User Prompt: Play devil's advocate: argue that Block's 10-K is DELIBERATELY
understating its Bitcoin and AI operational risks to avoid spooking investors.

Give 3 specific pieces of evidence from the disclosures.
Then give a counter-argument defending Block's disclosure adequacy.
Conclude with your net assessment.

CONTEXT:
=== BLOCK 10-K RISK FACTORS (excerpt) ===
Risk Factors 23Item 1B. Unresolved Staff Comments 61
Item 2. Properties 61Item 3. Legal Proceedings 61Item 4. Mine Safety Disclosures 61
PART IIItem 5. Market for Registrant’s Common Equity, Related Stockholder Matters and Issuer Purchases of Equity Securities 62Item 6. [RESERVED] 63
Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations 64Item 7A. Quantitative and Qualitative Disclosures About Market Risk 84Item 8. Financial Statements and Supplementa

In [8]:
#LLM-AS-A-JUDGE

judge_input = f"""
STRATEGY A OUTPUT:
{result_A[:1200]}

STRATEGY B OUTPUT:
{result_B[:1200]}

STRATEGY C OUTPUT:
{result_C[:1200]}
"""

result_judge = call_mistral(
    system_prompt="You are an impartial senior risk analyst acting as a judge.",
    user_prompt=f"""Three analysts used different prompting strategies to assess whether
Block's 10-K underplays AI and Bitcoin operational risks vs PayPal.

Evaluate each strategy on:
1. DEPTH — did it surface specific, non-obvious risks?
2. EVIDENCE — did it cite actual document language?
3. BALANCE — did it consider both sides?
4. USEFULNESS — would a compliance officer act on this?

Score each criterion 1–5. Declare a winner and explain why.

OUTPUTS:
{judge_input}"""
)

print("=== LLM-AS-A-JUDGE ===")
print(result_judge)

--- MISTRAL API CALL (placeholder) ---
System Prompt: You are an impartial senior risk analyst acting as a judge.
User Prompt: Three analysts used different prompting strategies to assess whether
Block's 10-K underplays AI and Bitcoin operational risks vs PayPal.

Evaluate each strategy on:
1. DEPTH — did it surface specific, non-obvious risks?
2. EVIDENCE — did it cite actual document language?
3. BALANCE — did it consider both sides?
4. USEFULNESS — would a compliance officer act on this?

Score each criterion 1–5. Declare a winner and explain why.

OUTPUTS:

STRATEGY A OUTPUT:
[Placeholder response from call_mistral]

STRATEGY B OUTPUT:
[Placeholder response from call_mistral]

STRATEGY C OUTPUT:
[Placeholder response from call_mistral]

=== LLM-AS-A-JUDGE ===
[Placeholder response from call_mistral]


In [ ]:
# EXECUTIVE SUMMARY

summary_context = f"""
BEST ANALYSIS:
{result_B[:800]}
{result_C[:600]}

JUDGE VERDICT:
{result_judge[:600]}
"""

executive_summary = call_mistral(
    system_prompt="You are a Chief Risk Officer writing for a board audience. Be concise, direct, and evidence-based.",
    user_prompt=f"""Based on the analysis below, write a 200-word Executive Summary answering:

'Is Block's AI and Bitcoin focus creating undisclosed operational risks
compared to PayPal's conservative strategy, and should investors be concerned?'

Requirements:
- Open with a clear verdict (Yes / No / Partially)
- Cite 2-3 specific risk gaps found in the 10-K comparison
- Reference Block's ecosystem bets (TBD, Tidal, Bitcoin mining)
- Close with a risk severity rating: Low / Medium / High and one recommended action

ANALYSIS:
{summary_context}"""
)

print("=" * 60)
print("EXECUTIVE SUMMARY — Block vs PayPal Risk Divergence")
print("=" * 60)
print(executive_summary)

--- MISTRAL API CALL (placeholder) ---
System Prompt: You are a Chief Risk Officer writing for a board audience. Be concise, direct, and evidence-based.
User Prompt: Based on the analysis below, write a 200-word Executive Summary answering:

'Is Block's AI and Bitcoin focus creating undisclosed operational risks
compared to PayPal's conservative strategy, and should investors be concerned?'

Requirements:
- Open with a clear verdict (Yes / No / Partially)
- Cite 2-3 specific risk gaps found in the 10-K comparison
- Reference Block's ecosystem bets (TBD, Tidal, Bitcoin mining)
- Close with a risk severity rating: Low / Medium / High and one recommended action

ANALYSIS:

BEST ANALYSIS:
[Placeholder response from call_mistral]
[Placeholder response from call_mistral]

JUDGE VERDICT:
[Placeholder response from call_mistral]

EXECUTIVE SUMMARY — Block vs PayPal Risk Divergence
[Placeholder response from call_mistral]
